# Session 14 · Case study: leaderboard round L2 with embeddings (and an optional language model)

**Block 3 practice.** Round L1 (Session 13) used TF-IDF alone. Round L2 asks: does a method of this session improve on it, by how much, and at what cost? This notebook

1. adds multilingual sentence embeddings (`intfloat/multilingual-e5-small`) to the TF-IDF features and measures the gain on the validation years, with a bootstrap interval for the difference;
2. optionally lets a language model choose among the ten best candidate headings for the requests where the trained model is unsure (abstention, theory page 03). **Without a model server this part is skipped and those requests keep the trained model's heading**, so the notebook always produces a submission;
3. refits on all 50,000 sample decisions and writes `submission-L2-embeddings.csv`.

**Run time.** Encoding the 50,000 sample and 113,188 test descriptions takes about 6 minutes with a GPU (including Apple silicon) and roughly 20–40 minutes on a CPU; the vectors are cached in `embeddings_cache/` next to this notebook, so later runs take about 4 minutes. Needs `sentence-transformers` (`uv sync --group embeddings`).

Theory: [03-generative-models-through-an-api.md](../theory/03-generative-models-through-an-api.md#practice-leaderboard-round-l2).

*Author: course team, licence CC-BY-4.0.*

In [ ]:
import time
from pathlib import Path

import numpy as np
import pandas as pd

# find the repository root (the folder that contains case-study/), wherever the notebook runs
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "case-study").exists())
DATA = ROOT / "case-study" / "data"
decisions = pd.read_parquet(DATA / "train_sample.parquet")
test = pd.read_parquet(DATA / "test.parquet")
nomenclature = pd.read_parquet(DATA / "nomenclature.parquet").set_index("heading")
fit = (decisions["start_date"].dt.year <= 2021).to_numpy()      # validation by time: 2022-2023
print(len(decisions), "sample decisions,", fit.sum(), "for fitting,", (~fit).sum(), "for validation;", len(test), "test")

## 1. Encode once, cache to disk

One vector of 384 numbers per description. The e5 models expect a prefix; for classification the model card recommends `query: `. Descriptions are cut at 256 tokens: the median description has about 170 tokens, 17 % are longer, and the cut halves the encoding time. The product is usually named at the beginning; the TF-IDF block still sees the whole text.

In [ ]:
from sentence_transformers import SentenceTransformer

MODEL_NAME = "intfloat/multilingual-e5-small"
encoder = SentenceTransformer(MODEL_NAME)
encoder.max_seq_length = 256
cache = Path("embeddings_cache")
cache.mkdir(exist_ok=True)


def embed(name, docs, prefix="query: "):
    """Encode once, then load from disk; the file name records model, prefix, length limit and data."""
    path = cache / f"{MODEL_NAME.split('/')[-1]}_{prefix.strip(': ')}_256_{name}.npy"
    if path.exists():
        return np.load(path)
    t0 = time.perf_counter()
    E = encoder.encode((prefix + docs).tolist(), batch_size=64, normalize_embeddings=True, show_progress_bar=False)
    print(f"{name}: {len(docs)} descriptions encoded in {time.perf_counter() - t0:.0f} s")
    np.save(path, E.astype(np.float32))
    return E


E_sample = embed("train_sample", decisions["description"])
E_test = embed("test", test["description"])
print(E_sample.shape, E_test.shape)

## 2. TF-IDF, embeddings and both, on the validation years

The same linear SVM as in round L1 on three feature sets. Both feature blocks have rows of length 1, so they can be placed side by side without rescaling; the weight of the embedding block is a setting to check on validation data.

In [ ]:
from scipy.sparse import csr_matrix, hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import accuracy_score, f1_score


def svm():
    return SGDClassifier(loss="hinge", alpha=1e-5, max_iter=20, tol=None, random_state=0, n_jobs=-1)


y_fit, y_val = decisions["heading"][fit].to_numpy(), decisions["heading"][~fit].to_numpy()
tfidf = TfidfVectorizer(min_df=2, sublinear_tf=True).fit(decisions["description"][fit])
T_fit, T_val = tfidf.transform(decisions["description"][fit]), tfidf.transform(decisions["description"][~fit])


def both(T, E, weight):
    return hstack([T, csr_matrix(weight * E)]).tocsr()


feature_sets = {
    "TF-IDF (round L1)": (T_fit, T_val),
    "embeddings": (E_sample[fit], E_sample[~fit]),
    "TF-IDF + embeddings, weight 0.5": (both(T_fit, E_sample[fit], 0.5), both(T_val, E_sample[~fit], 0.5)),
    "TF-IDF + embeddings, weight 1": (both(T_fit, E_sample[fit], 1.0), both(T_val, E_sample[~fit], 1.0)),
    "TF-IDF + embeddings, weight 2": (both(T_fit, E_sample[fit], 2.0), both(T_val, E_sample[~fit], 2.0)),
}
rows, preds, kept = [], {}, {}
for name, (A_fit, A_val) in feature_sets.items():
    t0 = time.perf_counter()
    clf = svm().fit(A_fit, y_fit)
    preds[name] = clf.predict(A_val)
    acc = accuracy_score(y_val, preds[name])
    if name.startswith("TF-IDF +") and acc > kept.get("accuracy", 0):   # keep only the best combination
        kept = {"name": name, "clf": clf, "A_val": A_val, "accuracy": acc}  # (each model takes about 0.5 GB)
    rows.append({"features": name, "accuracy": acc, "macro_F1": f1_score(y_val, preds[name], average="macro"),
                 "seconds": time.perf_counter() - t0})
results = pd.DataFrame(rows).set_index("features")
results.round(3)

Is the gain of the best combination larger than the noise? A **paired bootstrap** resamples the validation decisions and recomputes the *difference* in accuracy, so both models are always compared on the same decisions.

In [ ]:
best = kept["name"]
right_best = preds[best] == y_val
right_l1 = preds["TF-IDF (round L1)"] == y_val
rng = np.random.default_rng(0)
diffs = []
for _ in range(2000):
    i = rng.integers(0, len(y_val), len(y_val))           # the same resampled decisions for both models
    diffs.append(right_best[i].mean() - right_l1[i].mean())
print(f"{best}: {right_best.mean() - right_l1.mean():+.3f} accuracy against round L1, "
      f"95 % interval [{np.percentile(diffs, 2.5):+.3f}, {np.percentile(diffs, 97.5):+.3f}]")
print(pd.crosstab(right_l1, right_best, rownames=["L1 right"], colnames=[f"{best} right"]))

**Question.** The combination is right on some decisions where TF-IDF fails and wrong on others where TF-IDF was right. Read five of each kind (`decisions[~fit][right_best & ~right_l1]`): are they in particular languages?

## 3. Optional: a language model for the unsure requests — REQUIRES A MODEL SERVER

The SVM's highest score says how convinced it is: below 0 it is not convinced of any heading (theory page 03, abstention). For those requests a language model may choose among the ten best candidates. The cells below use the setup of workbook 09 (Ollama by default; `LLM_BASE_URL`, `LLM_MODEL`, `LLM_API_KEY` for another provider). **Without a server they are skipped, and every request keeps the trained model's heading.**

In [ ]:
clf_best, A_val_best, pred_best = kept["clf"], kept["A_val"], preds[best]
scores_val = clf_best.decision_function(A_val_best)
top_score = scores_val.max(axis=1)
unsure = top_score < 0
print(f"unsure requests (top score < 0): {unsure.mean():.1%} of validation; "
      f"accuracy there {np.mean(pred_best[unsure] == y_val[unsure]):.3f}, elsewhere {np.mean(pred_best[~unsure] == y_val[~unsure]):.3f}")
top10_val = clf_best.classes_[np.argsort(-scores_val, axis=1)[:, :10]]
print("true heading among the 10 candidates of the unsure requests:",
      round(np.mean([h in row for h, row in zip(y_val[unsure], top10_val[unsure])]), 3))

In [ ]:
import os

import httpx
from openai import OpenAI
from pydantic import BaseModel, Field

BASE_URL = os.getenv("LLM_BASE_URL", "http://localhost:11434/v1")
MODEL = os.getenv("LLM_MODEL", "qwen2.5:3b")
API_KEY = os.getenv("LLM_API_KEY", "ollama")
try:
    LLM_AVAILABLE = httpx.get(BASE_URL.rstrip("/") + "/models", headers={"Authorization": f"Bearer {API_KEY}"},
                              timeout=3).status_code == 200
except httpx.HTTPError:
    LLM_AVAILABLE = False
print(BASE_URL, MODEL, "server reachable:", LLM_AVAILABLE)


class HeadingChoice(BaseModel):
    heading: str = Field(pattern=r"^\d{4}$")


SYSTEM = ("You are an assistant for EU customs classification. You receive the description of goods of a "
          "Binding Tariff Information request, in any EU language, and candidate HS headings with their English "
          'texts. Choose the single candidate that fits the goods best. Answer only with JSON {"heading": "<four digits>"}.')


def llm_choice(client, description, candidates, fallback):
    """The language model's choice among the candidates; the trained model's heading if anything goes wrong."""
    lines = "\n".join(f"{h}: {nomenclature['heading_description'].get(h, '')[:150]}" for h in candidates)
    schema = HeadingChoice.model_json_schema()
    schema["properties"]["heading"]["enum"] = list(candidates)
    try:
        resp = client.chat.completions.create(
            model=MODEL, temperature=0,
            messages=[{"role": "system", "content": SYSTEM},
                      {"role": "user", "content": f"Description of goods:\n{description[:3000]}\n\nCandidate headings:\n{lines}"}],
            response_format={"type": "json_schema", "json_schema": {"name": "HeadingChoice", "schema": schema}})
        heading = HeadingChoice.model_validate_json(resp.choices[0].message.content).heading
        return heading if heading in candidates else fallback
    except Exception:
        return fallback


N_LLM_VALID = 200                      # unsure validation requests sent to the model (seeded sample)
if LLM_AVAILABLE:
    client = OpenAI(base_url=BASE_URL, api_key=API_KEY, timeout=60)
    idx = np.random.default_rng(0).choice(np.flatnonzero(unsure), size=min(N_LLM_VALID, unsure.sum()), replace=False)
    texts_val = decisions["description"][~fit].to_numpy()
    t0 = time.perf_counter()
    llm_pred = np.array([llm_choice(client, texts_val[i], list(top10_val[i]), pred_best[i]) for i in idx])
    sec = (time.perf_counter() - t0) / len(idx)
    print(f"{len(idx)} unsure requests: trained model {np.mean(pred_best[idx] == y_val[idx]):.3f}, "
          f"LLM {np.mean(llm_pred == y_val[idx]):.3f}, {sec:.1f} s per request")
else:
    print("No model server: the LLM step is skipped; the submission uses the trained model for all requests.")

**Decision.** Use the language model in the submission only if it beats the trained model on the unsure validation requests by more than the noise (200 requests: about ±7 points), and only if you can afford the calls (time per request × unsure test requests).

## 4. Refit on all sample decisions and predict the test set

In [ ]:
USE_LLM_ON_TEST = False               # set to True only after the validation in section 3 supports it
MAX_LLM_CALLS = 2_000                 # the most unsure test requests sent to the model

weight = {"TF-IDF + embeddings, weight 0.5": 0.5, "TF-IDF + embeddings, weight 1": 1.0,
          "TF-IDF + embeddings, weight 2": 2.0}[best]
tfidf_all = TfidfVectorizer(min_df=2, sublinear_tf=True).fit(decisions["description"])
A_all = both(tfidf_all.transform(decisions["description"]), E_sample, weight)
A_test = both(tfidf_all.transform(test["description"]), E_test, weight)
final = svm().fit(A_all, decisions["heading"])
scores_test = final.decision_function(A_test)
heading = final.classes_[scores_test.argmax(axis=1)]
print(f"unsure test requests (top score < 0): {np.mean(scores_test.max(axis=1) < 0):.1%}")

if USE_LLM_ON_TEST and LLM_AVAILABLE:
    order = np.argsort(scores_test.max(axis=1))[:MAX_LLM_CALLS]          # least convinced first
    top10_test = final.classes_[np.argsort(-scores_test[order], axis=1)[:, :10]]
    for j, i in enumerate(order):
        heading[i] = llm_choice(client, test["description"].iloc[i], list(top10_test[j]), heading[i])
    method = f"TF-IDF + e5-small embeddings, {MODEL} on the {len(order)} least certain requests"
else:
    method = "TF-IDF + e5-small embeddings (linear SVM)"
submission = pd.DataFrame({"id": test["id"], "heading": heading})
assert len(submission) == len(test) and submission["id"].is_unique
assert submission["heading"].str.fullmatch(r"\d{4}").all()
path = Path("submission-L2-embeddings.csv")
submission.to_csv(path, index=False)
print(method, "->", path.resolve())

## 5. Score

Upload the file to the course leaderboard, or score it locally where the solution file is available (lecturer):

```bash
uv run python case-study/score.py sessions/14-llms-and-embeddings/workbooks/submission-L2-embeddings.csv
```

The next cell does the same if `case-study/data/instructor/solution.csv` exists. Expected with the default path: 0.824 public and 0.804 private accuracy (macro-F1 0.527 and 0.502), against 0.803 and 0.776 for the word TF-IDF model of round L1 on the same sample.

In [ ]:
import sys

solution = DATA / "instructor" / "solution.csv"
if solution.exists():
    sys.path.insert(0, str(ROOT / "case-study"))
    from score import score
    for k, v in score(submission, pd.read_csv(solution, dtype=str)).items():
        print(f"{k:22s} {v:.4f}")
else:
    print("No solution file: upload the CSV to the leaderboard instead.")

## Your turn

1. Write the L2 entry of your team log: method, settings, validation accuracy and macro-F1 with the paired interval against L1, leaderboard scores, encoding time and, if used, LLM calls and seconds per request.
2. Replace the word TF-IDF block by character n-grams (`analyzer="char_wb", ngram_range=(3, 5), min_df=3, max_features=300_000`, about 3 more minutes). Does the embedding block still add something on top of a stronger TF-IDF model?
3. With a model server: is the LLM better than the trained model on the unsure requests? Compute what it would cost in time to send all unsure test requests, and decide whether L2 should use it.
4. Delete the local submission file after uploading it.

In [ ]:
# your code here